In [ ]:
! pip install pyTelegramBotAPI schedule pytz
import telebot
from telebot import types
import json
import os
import random
import threading
import time
import pytz
from datetime import datetime, timedelta

# CONFIG =======================================================================
TOKEN = "8791421129:AAFcnSokvEbxF0AkP7atKC-7RFvXZJTQFcw"
bot = telebot.TeleBot(TOKEN)
DATA_FILE = "finance_data.json"
KZ_TIMEZONE = pytz.timezone("Asia/Almaty")

budget_flow = {}
todo_flow = {}

CATEGORIES = [
    "🎭 Entertainment", "☕ Cafe & Food", "🛒 Shopping",
    "🚌 Transport", "🏠 Bills", "⚙️ Other"
]

# DATA ENGINE ==================================================================
def load_data():
    if not os.path.exists(DATA_FILE): return {}
    try:
        with open(DATA_FILE, "r", encoding="utf-8") as f: return json.load(f)
    except: return {}

def save_data(data):
    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def get_user_profile(data, user_id):
    uid = str(user_id)
    if uid not in data:
        data[uid] = {"settings": {"budget_limit": 0}, "transactions": [], "todos": []}
    return data, uid

def clean_number(text):
    return float(text.replace(" ", "").replace(",", "."))

# KEYBOARDS ====================================================================
def main_menu():
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True, row_width=2)
    markup.add("💸 Add Expense", "💰 Add Income")
    markup.add("📊 Statistics", "🧠 Budget Plan")
    markup.add("🏠 Mortgage Calc", "📅 Daily Summary")
    markup.add("📝 To-Do Reminder")
    return markup

def back_kbd():
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.add("🔙 Back to Menu")
    return markup

# ADVICE ENGINE ================================================================
def get_ai_advice(uid_data, current_category=None):
    transactions = uid_data.get("transactions", [])
    if not transactions:
        return "💡 Start tracking your expenses to receive personalized advice."

    cat_total = sum(t["amount"] for t in transactions if t.get("category") == current_category)

    if current_category == "🛒 Shopping" and cat_total > 50000:
        return "⚠️ Your shopping spending is high this period. Try to stick to a strict list next time!"
    if current_category == "🎭 Entertainment" and cat_total > 30000:
        return "🎉 You've had a lot of fun lately! Consider a 'no-spend' weekend to balance it out."

    tips = [
        "💰 Saving 20% of your income is the golden rule of finance.",
        "🏦 Banks in Kazakhstan often have great cashback—check your app!",
        "📈 Try the 50/30/20 rule: 50% for Needs, 30% for Wants, 20% for Savings.",
        "💡 Small habits like daily coffee can cost over 200,000 ₸ a year.",
        "📊 Use the 'Month' statistics regularly to track your progress."
    ]
    return random.choice(tips)

# START & BACK HANDLER =========================================================
@bot.message_handler(commands=['start'])
@bot.message_handler(func=lambda m: m.text == "🔙 Back to Menu")
def start(message):
    bot.send_message(message.chat.id, "✨ FinMate AI Menu:", reply_markup=main_menu())

# 1. INCOME ====================================================================
@bot.message_handler(func=lambda m: m.text == "💰 Add Income")
def inc_init(message):
    msg = bot.send_message(message.chat.id, "💵 Enter income amount:", reply_markup=back_kbd())
    bot.register_next_step_handler(msg, save_inc)

def save_inc(message):
    if message.text == "🔙 Back to Menu": return start(message)
    try:
        val = clean_number(message.text)
        data = load_data()
        data, uid = get_user_profile(data, message.chat.id)
        now = datetime.now(KZ_TIMEZONE).strftime("%Y-%m-%d %H:%M:%S")
        data[uid]["transactions"].append({"type": "income", "amount": val, "timestamp": now})
        save_data(data)
        bot.send_message(message.chat.id, f"✅ Saved: {val:,.0f} ₸", reply_markup=main_menu())
    except:
        bot.register_next_step_handler(bot.send_message(message.chat.id, "❌ Invalid number. Try again:"), save_inc)

# 2. EXPENSE ===================================================================
@bot.message_handler(func=lambda m: m.text == "💸 Add Expense")
def exp_init(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True, row_width=2)
    markup.add(*[types.KeyboardButton(c) for c in CATEGORIES])
    markup.add("🔙 Back to Menu")
    bot.send_message(message.chat.id, "📂 Select category:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in CATEGORIES)
def exp_val_step(message):
    cat = message.text
    msg = bot.send_message(message.chat.id, f"How much for {cat}?", reply_markup=back_kbd())
    bot.register_next_step_handler(msg, save_expense, cat)

def save_expense(message, category):
    # BACK BUTTON CHECK
    if message.text == "🔙 Back to Menu":
        return start(message)

    try:
        # Санын тазалап алу
        amount = clean_number(message.text)

        # Мәліметтерді жүктеу
        data = load_data()
        data, uid = get_user_profile(data, message.chat.id)

        # Қазіргі уақыт
        now = datetime.now(KZ_TIMEZONE).strftime("%Y-%m-%d %H:%M:%S")

        # Тізімге қосу
        new_transaction = {
            "type": "expense",
            "amount": amount,
            "category": category,
            "timestamp": now
        }
        data[uid]["transactions"].append(new_transaction)

        # Файлға сақтау
        save_data(data)

        # Кеңес алу (advice)
        advice = get_ai_advice(data[uid], category)

        # Жауап мәтіні
        res = (
          f"✅ *Recorded successfully!*\n"
          + "—" * 15 + "\n"
          + f"📂 Category: {category}\n"
          + f"💸 Amount: {amount:,.0f} ₸\n\n"
          + f"🤖 *AI Advice:* {advice}"
        )

        bot.send_message(message.chat.id, res, parse_mode="Markdown", reply_markup=main_menu())

    except Exception as e:

        print(f"Error: {e}")
        msg = bot.send_message(message.chat.id, "❌ Invalid number. Please enter digits only:")
        bot.register_next_step_handler(msg, save_expense, category)

# 3. STATISTICS ================================================================
@bot.message_handler(func=lambda m: m.text == "📊 Statistics")
def stats_init(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.add("Day", "Week", "Month", "Year", "🔙 Back to Menu")
    bot.send_message(message.chat.id, "Select Period:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in ["Day", "Week", "Month", "Year"])
def show_detailed_stats(message):
    period = message.text
    data = load_data()
    data, uid = get_user_profile(data, message.chat.id)
    now = datetime.now(KZ_TIMEZONE)
    deltas = {"Day": timedelta(days=1), "Week": timedelta(weeks=1), "Month": timedelta(days=30), "Year": timedelta(days=365)}
    start_date = now - deltas[period]
    date_range = f"{start_date.strftime('%d.%m')} — {now.strftime('%d.%m')}"

    cat_totals = {}; inc_total, exp_total = 0, 0
    for t in data[uid]["transactions"]:
        t_date = KZ_TIMEZONE.localize(datetime.strptime(t["timestamp"], "%Y-%m-%d %H:%M:%S"))
        if t_date >= start_date:
            if t["type"] == "income": inc_total += t["amount"]
            else:
                exp_total += t["amount"]
                cat = t.get("category", "⚙️ Other")
                cat_totals[cat] = cat_totals.get(cat, 0) + t["amount"]

    res = f"📊 *{period.upper()} REPORT*\n📅 `{date_range}`\n" + "—"*15 + "\n"
    for c, val in cat_totals.items(): res += f"{c}: {val:,.0f} ₸\n"
    res += f"\n📉 Expenses: {exp_total:,.0f} ₸\n📈 Income: {inc_total:,.0f} ₸\n💎 Balance: {inc_total-exp_total:,.0f} ₸"
    bot.send_message(message.chat.id, res, parse_mode="Markdown", reply_markup=main_menu())

# 5. BUDGET PLANNER (WITH BACK BUTTON) =========================================
@bot.message_handler(func=lambda m: m.text == "🧠 Budget Plan")
def budget_start(message):
    msg = bot.send_message(message.chat.id, "💰 Enter monthly income:", reply_markup=back_kbd())
    bot.register_next_step_handler(msg, budget_income)

def budget_income(message):
    if message.text == "🔙 Back to Menu": return start(message)
    try:
        income = clean_number(message.text)
        budget_flow[message.chat.id] = {"income": income}
        msg = bot.send_message(message.chat.id, "🏠 Enter rent/mortgage:", reply_markup=back_kbd())
        bot.register_next_step_handler(msg, budget_rent)
    except:
        bot.register_next_step_handler(bot.send_message(message.chat.id, "❌ Try again:"), budget_income)

def budget_rent(message):
    if message.text == "🔙 Back to Menu": return start(message)
    try:
        budget_flow[message.chat.id]["rent"] = clean_number(message.text)
        msg = bot.send_message(message.chat.id, "💡 Enter utilities:", reply_markup=back_kbd())
        bot.register_next_step_handler(msg, budget_utilities)
    except:
        bot.register_next_step_handler(bot.send_message(message.chat.id, "❌ Try again:"), budget_rent)

def budget_utilities(message):
    if message.text == "🔙 Back to Menu": return start(message)
    try:
        budget_flow[message.chat.id]["utilities"] = clean_number(message.text)
        msg = bot.send_message(message.chat.id, "🚕 Enter transport costs:", reply_markup=back_kbd())
        bot.register_next_step_handler(msg, budget_transport)
    except:
        bot.register_next_step_handler(bot.send_message(message.chat.id, "❌ Try again:"), budget_utilities)

def budget_transport(message):
    if message.text == "🔙 Back to Menu": return start(message)
    try:
        budget_flow[message.chat.id]["transport"] = clean_number(message.text)
        msg = bot.send_message(message.chat.id, "📦 Enter other fixed costs:", reply_markup=back_kbd())
        bot.register_next_step_handler(msg, budget_result)
    except:
        bot.register_next_step_handler(bot.send_message(message.chat.id, "❌ Try again:"), budget_transport)

def budget_result(message):
    if message.text == "🔙 Back to Menu": return start(message)
    try:
        other = clean_number(message.text)
        d = budget_flow[message.chat.id]
        fixed = d["rent"] + d["utilities"] + d["transport"] + other
        rem = d["income"] - fixed
        res = (f"🧠 *SMART BUDGET PLAN*\n\n💰 Income: {d['income']:,.0f}\n📌 Fixed: {fixed:,.0f}\n📈 Left: {rem:,.0f}\n\n"
               f"🛒 Spending: {rem*0.4:,.0f}\n🎉 Fun: {rem*0.2:,.0f}\n💾 Savings: {rem*0.2:,.0f}\n🏦 Deposit: {rem*0.2:,.0f}")
        bot.send_message(message.chat.id, res, parse_mode="Markdown", reply_markup=main_menu())
    except:
        bot.register_next_step_handler(bot.send_message(message.chat.id, "❌ Error:"), budget_result)


# 4. MORTGAGE ==================================================================
@bot.message_handler(func=lambda m: m.text == "🏠 Mortgage Calc")
def mort_init(message):
    msg = bot.send_message(message.chat.id, "Enter: Amount Interest Years (e.g. 20000000 14.5 20)", reply_markup=back_kbd())
    bot.register_next_step_handler(msg, mort_res)

def mort_res(message):
    if message.text == "🔙 Back to Menu": return start(message)
    try:
        p = message.text.split()
        amt, rate, yrs = float(p[0]), float(p[1])/100/12, int(p[2])*12
        m_pay = amt * (rate * (1+rate)**yrs) / ((1+rate)**yrs - 1)
        res = f"Monthly: {m_pay:,.0f} ₸\nTotal: {m_pay*yrs:,.0f} ₸"
        bot.send_message(message.chat.id, res, reply_markup=main_menu())
    except: bot.send_message(message.chat.id, "❌ Format error", reply_markup=main_menu())


# 6. DAILY SUMMARY =============================================================
@bot.message_handler(func=lambda m: m.text == "📅 Daily Summary")
def daily_sum(message):
    data = load_data()
    data, uid = get_user_profile(data, message.chat.id)
    today = datetime.now(KZ_TIMEZONE).strftime("%Y-%m-%d")
    spent = sum(t["amount"] for t in data[uid]["transactions"] if t["timestamp"].startswith(today) and t["type"]=="expense")
    bot.send_message(message.chat.id, f"📅 Spent today: {spent:,.0f} ₸", reply_markup=main_menu())

# 7. TO-DO SYSTEM ==============================================================
@bot.message_handler(func=lambda m: m.text == "📝 To-Do Reminder")
def todo_init(message):
    msg = bot.send_message(message.chat.id, "📝 Task description:", reply_markup=back_kbd())
    bot.register_next_step_handler(msg, todo_dl)

def todo_dl(message):
    if message.text == "🔙 Back to Menu": return start(message)
    todo_flow[message.chat.id] = {"task": message.text}
    msg = bot.send_message(message.chat.id, "📅 Deadline (YYYY-MM-DD):", reply_markup=back_kbd())
    bot.register_next_step_handler(msg, todo_t)

def todo_t(message):
    if message.text == "🔙 Back to Menu": return start(message)
    todo_flow[message.chat.id]["deadline"] = message.text
    msg = bot.send_message(message.chat.id, "⏰ Time (HH:MM):", reply_markup=back_kbd())
    bot.register_next_step_handler(msg, todo_rep)

def todo_rep(message):
    if message.text == "🔙 Back to Menu": return start(message)
    todo_flow[message.chat.id]["time"] = message.text
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.add("Every Hour", "Every 5 Hours", "Every Day", "🔙 Back to Menu")
    msg = bot.send_message(message.chat.id, "🔁 Frequency:", reply_markup=markup)
    bot.register_next_step_handler(msg, todo_save)

def todo_save(message):
    if message.text == "🔙 Back to Menu": return start(message)
    data = load_data()
    data, uid = get_user_profile(data, message.chat.id)
    flow = todo_flow[message.chat.id]
    flow["repeat"] = message.text
    data[uid]["todos"].append(flow)
    save_data(data)
    bot.send_message(message.chat.id, "✅ Saved", reply_markup=main_menu())



# RUN ==========================================================================
print("FinMate AI is running...")
bot.infinity_polling()

FinMate AI is running...


2026-05-11 17:48:20,290 (__init__.py:1254 MainThread) ERROR - TeleBot: "Threaded polling exception: A request to the Telegram API was unsuccessful. Error code: 409. Description: Conflict: terminated by other getUpdates request; make sure that only one bot instance is running"
ERROR:TeleBot:Threaded polling exception: A request to the Telegram API was unsuccessful. Error code: 409. Description: Conflict: terminated by other getUpdates request; make sure that only one bot instance is running
2026-05-11 17:48:20,292 (__init__.py:1256 MainThread) ERROR - TeleBot: "Exception traceback:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/telebot/__init__.py", line 1247, in __threaded_polling
    polling_thread.raise_exceptions()
  File "/usr/local/lib/python3.12/dist-packages/telebot/util.py", line 116, in raise_exceptions
    raise self.exception_info
  File "/usr/local/lib/python3.12/dist-packages/telebot/util.py", line 98, in run
    task(*args, **kwargs)
  